# Simple Sorter Gate Validation

Checks whether `run_sorter('simple')` correctly gates a real recording,
and validates that a synthetic silent recording returns 0 units.

Use this to tune the detection threshold before relying on the gate in `run_shank.py`.

In [ ]:
from pathlib import Path
import numpy as np
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
from spikeinterface.extractors.neuropixels_utils import get_neuropixels_sample_shifts
import sys
sys.path.insert(0, str(Path('.').resolve()))
from probe_utils import load_probe
from get_artifacts import detect_saturation_periods

## Configuration — edit these

In [ ]:
FOLDER        = Path('/path/to/session/folder')  # top-level session folder
PROBE         = 'A'                               # probe letter, e.g. 'A'
SHANK_NUM     = '0'                               # shank to test
SAMPLE_RATE   = 30000
N_CHANNELS_PROBE = 384

# How many seconds to use for the check (shorter = faster)
TEST_DURATION_SEC = 60

# Detection threshold passed to the simple sorter (default is 5.0 std)
DETECT_THRESHOLD = 5.0

# --- derived paths ---
data_folder  = FOLDER / 'data'
probe_file   = FOLDER / f'{PROBE}_probe_conf.json'
scratch_dir  = FOLDER / 'output' / PROBE / f'shank_{SHANK_NUM}' / 'simple_sorter_check'
scratch_dir.mkdir(parents=True, exist_ok=True)

## Load & preprocess the recording (same steps as run_shank.py)

In [ ]:
import glob

probe_name = f'np2-{PROBE}-ephys'
recording_files = sorted(glob.glob(f'{data_folder}/{probe_name}*'))
print(f'Found {len(recording_files)} file(s):', recording_files)

probe_data, _    = load_probe(probe_file)
shank_probe, N_CHANNELS_SHANK = load_probe(probe_file, SHANK_NUM)
print(f'Shank {SHANK_NUM} has {N_CHANNELS_SHANK} channels')

In [ ]:
recordings = []
for f in recording_files:
    rec = se.read_binary(f, dtype='int16', sampling_frequency=SAMPLE_RATE,
                         num_channels=N_CHANNELS_PROBE)
    # Trim to TEST_DURATION_SEC to keep things fast
    n_frames = min(rec.get_num_frames(), int(SAMPLE_RATE * TEST_DURATION_SEC))
    rec = rec.frame_slice(0, n_frames)

    sat_idx = detect_saturation_periods(rec, abs_threshold=3900, direction='upper',
                                        chunk_size=30000 * 10, n_jobs=4)
    rec = si.remove_artifacts(rec, list_triggers=sat_idx, ms_before=10, ms_after=10,
                               mode='zeros')
    recordings.append(rec)

total_rec = si.concatenate_recordings(recordings)
total_rec.set_probe(probe_data)
total_rec.set_channel_locations(probe_data.contact_positions)

total_rec = si.highpass_filter(total_rec, ftype='bessel', dtype='float32')

sample_shifts = get_neuropixels_sample_shifts(384, 16, 16)
total_rec = spre.phase_shift(total_rec, inter_sample_shift=sample_shifts)
total_rec.set_property('group', probe_data.shank_ids)

rec_split = total_rec.split_by('group')
shank_key = str(SHANK_NUM)
if shank_key not in rec_split:
    shank_key = int(SHANK_NUM)
rec = rec_split[shank_key]

rec = si.highpass_spatial_filter(rec, dtype='int16')
rec = rec.set_probe(shank_probe)

rec.set_channel_gains(1)
rec.set_channel_offsets(0)

_, ch_dead  = si.detect_bad_channels(rec, method='coherence+psd', seed=42)
_, ch_noise = si.detect_bad_channels(rec, method='mad', seed=42)
bad_ids = np.concatenate([
    rec.get_channel_ids()[ch_dead  == 'dead'],
    rec.get_channel_ids()[ch_noise == 'noise'],
])
if bad_ids.size > 0:
    rec = si.interpolate_bad_channels(rec, bad_ids)

rec = si.common_reference(rec, reference='local')
shank_rec = spre.bandpass_filter(rec, freq_min=300., freq_max=7500., dtype='int16')

print('Preprocessed recording:', shank_rec)
print(f'Duration: {shank_rec.get_total_duration():.1f} s, '
      f'{shank_rec.get_num_channels()} channels')

## Run simple sorter on the real recording

In [ ]:
real_sort_dir = scratch_dir / 'real'

sorting_real = si.run_sorter(
    sorter_name='simple',
    recording=shank_rec,
    output_folder=real_sort_dir,
    remove_existing_folder=True,
    verbose=True,
    sorter_params={'detect_threshold': DETECT_THRESHOLD},
)

unit_ids = sorting_real.get_unit_ids()
print(f'\nUnits found: {len(unit_ids)}')

In [ ]:
import pandas as pd

if len(unit_ids) > 0:
    rows = []
    for uid in unit_ids:
        spikes = sorting_real.get_unit_spike_train(uid, segment_index=0)
        rows.append({
            'unit_id': uid,
            'n_spikes': len(spikes),
            'firing_rate_hz': len(spikes) / shank_rec.get_total_duration(),
        })
    df = pd.DataFrame(rows).sort_values('firing_rate_hz', ascending=False)
    print(df.to_string(index=False))
    print(f'\nTotal spikes across all units: {df["n_spikes"].sum()}')
else:
    print('Gate would STOP processing (0 units found).')

## Sanity check: synthetic silent recording → must return 0 units

Replaces every sample with Gaussian noise at a level well below the detection
threshold so we can confirm the gate would not false-positive on a dead channel.

In [ ]:
from spikeinterface.core import NumpyRecording

n_ch  = shank_rec.get_num_channels()
n_fr  = shank_rec.get_num_frames()
rng   = np.random.default_rng(0)

# Low-amplitude noise (~1 µV equivalent) — should produce no threshold crossings
silent_traces = rng.normal(0, 1, size=(n_fr, n_ch)).astype('int16')
silent_rec = NumpyRecording(silent_traces, sampling_frequency=SAMPLE_RATE)
silent_rec = silent_rec.set_probe(shank_probe)
silent_rec.set_channel_gains(1)
silent_rec.set_channel_offsets(0)

silent_sort_dir = scratch_dir / 'silent'
sorting_silent = si.run_sorter(
    sorter_name='simple',
    recording=silent_rec,
    output_folder=silent_sort_dir,
    remove_existing_folder=True,
    verbose=False,
    sorter_params={'detect_threshold': DETECT_THRESHOLD},
)

n_silent_units = len(sorting_silent.get_unit_ids())
print(f'Silent recording → {n_silent_units} units detected')
if n_silent_units == 0:
    print('PASS: gate would correctly stop on a silent recording.')
else:
    print('WARNING: gate would NOT stop on a silent recording — consider raising DETECT_THRESHOLD.')

## Summary

In [ ]:
gate_decision = 'PASS (continue to Kilosort)' if len(unit_ids) > 0 else 'STOP (write NO_GOOD_SPIKES)'

print('=== Simple Sorter Gate Summary ===')
print(f'  Recording : {FOLDER.name} / probe {PROBE} / shank {SHANK_NUM}')
print(f'  Duration tested : {shank_rec.get_total_duration():.1f} s')
print(f'  Detect threshold : {DETECT_THRESHOLD} std')
print(f'  Units on real recording : {len(unit_ids)}')
print(f'  Units on silent recording : {n_silent_units}')
print(f'  Gate decision : {gate_decision}')